<a href="https://colab.research.google.com/github/praut05/Github/blob/main/Column_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('covid_toy.csv')

In [ ]:
df.sample(10)

,age,gender,fever,cough,city,has_covid
33,26,Female,98.0,Mild,Kolkata,No
28,16,Male,104.0,Mild,Kolkata,No
75,5,Male,102.0,Mild,Kolkata,Yes
13,64,Male,102.0,Mild,Bangalore,Yes
27,33,Female,102.0,Strong,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
55,81,Female,101.0,Mild,Mumbai,Yes
18,64,Female,98.0,Mild,Bangalore,Yes
2,42,Male,101.0,Mild,Delhi,No
93,27,Male,100.0,Mild,Kolkata,Yes


In [ ]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [ ]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [ ]:
df['gender'].unique()

array(['Male', 'Female'], dtype=object)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [ ]:
X_train

,age,gender,fever,cough,city
73,34,Male,98.0,Strong,Kolkata
78,11,Male,100.0,Mild,Bangalore
1,27,Male,100.0,Mild,Delhi
83,17,Female,104.0,Mild,Kolkata
3,31,Female,98.0,Mild,Kolkata
...,...,...,...,...,...
51,11,Female,100.0,Strong,Kolkata
84,69,Female,98.0,Strong,Mumbai
39,50,Female,103.0,Mild,Kolkata
55,81,Female,101.0,Mild,Mumbai


In [ ]:
y_train

,has_covid
73,Yes
78,Yes
1,Yes
83,No
3,No
...,...
51,Yes
84,No
39,No
55,Yes


1. Aam Zindagi

In [ ]:
# adding simple imputer to fever col (basically it will replace all missing to mean)
from sklearn.impute import SimpleImputer
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also to test data
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [ ]:
X_train_fever

array([[ 98.        ],
       [100.        ],
       [100.        ],
       [104.        ],
       [ 98.        ],
       [102.        ],
       [100.        ],
       [101.        ],
       [102.        ],
       [103.        ],
       [102.        ],
       [101.        ],
       [100.        ],
       [104.        ],
       [ 98.        ],
       [100.        ],
       [104.        ],
       [101.        ],
       [104.        ],
       [100.        ],
       [100.        ],
       [104.        ],
       [102.        ],
       [ 99.        ],
       [ 98.        ],
       [103.        ],
       [ 99.        ],
       [ 99.        ],
       [100.66666667],
       [ 98.        ],
       [103.        ],
       [ 98.        ],
       [101.        ],
       [100.66666667],
       [101.        ],
       [ 99.        ],
       [104.        ],
       [ 98.        ],
       [104.        ],
       [100.66666667],
       [104.        ],
       [100.        ],
       [ 98.        ],
       [ 98

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
# Ordinal encoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also for test
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also for test
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [ ]:
# Extracting age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also for test
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

Join all the columns

In [ ]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

2. Mentos Zindagi

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [ ]:
transformer

ColumnTransformer(remainder='passthrough',
                  transformers=[('tnf1', SimpleImputer(), ['fever']),
                                ('tnf2',
                                 OrdinalEncoder(categories=[['Mild',
                                                             'Strong']]),
                                 ['cough']),
                                ('tnf3',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['gender', 'city'])])

In [ ]:
transformer.fit_transform(X_train).shape

(80, 7)

In [ ]:
transformer.fit_transform(X_test).shape

(20, 7)